In [ ]:
import pandas as pd

#### diagnoses are icd 10 codes, 3 digit resolution
there's columns for date of appearance, for htose with it, per code in order. Seperated by "|"

In [ ]:
df_diag = pd.read_csv("diagnosis.csv.gz")
df_diag

In [ ]:
## cholesterol nmr df - filter by this
filt_df = df = pd.read_csv("Cholesterol_miki_participant.csv.gz").dropna(thresh=5,axis=0)
print(filt_df.shape)
filt_ids = list(filt_df["Participant ID"].unique())
print(len(filt_ids),"# filt IDs")

In [ ]:
df_diag = df_diag.loc[df_diag['Participant ID'].isin(filt_ids)]
df_diag.shape[0]

In [ ]:
df_diag.columns

In [ ]:
df_diag["Year of birth"] = df_diag['Year of birth(participant - p34)'].fillna(df_diag['Year of birth(participant - p22200)'])

df_diag['Diagnoses - ICD10'].fillna("",inplace=True)

In [ ]:
df_diag = df_diag[['Participant ID', "Year of birth",
                   'Sex', 'Diagnoses - ICD10']].drop_duplicates()

df_diag

In [ ]:
df_diag.nunique()

In [ ]:
df_diag.isna().max()

In [ ]:
df_diag[df_diag["Diagnoses - ICD10"].str.contains("E78.01",case=False,na=False)]# no res

In [ ]:
df_diag[df_diag["Diagnoses - ICD10"].str.contains("E78.0",case=False,na=False)]

In [ ]:
df_diag[df_diag["Diagnoses - ICD10"].str.contains("Familial hyper",case=False,na=False)]

In [ ]:
df_diag.dropna()

In [ ]:
df_diag[df_diag["Diagnoses - ICD10"].str.contains("I25",case=False,na=False)]

In [ ]:
# df = df_diag.head(5000).copy()

df = df_diag.copy()

In [ ]:
%%time
# Re-implementing the process without explicitly using an extra 'index' column

K_value = 100#2_500

# Start by creating a long format dataframe of diagnoses directly from the original dataframe, preserving the index
diagnoses_long_df_refined = df['Diagnoses - ICD10'].str.split('|', expand=True).stack().reset_index(level=1, drop=True).to_frame('Diagnosis')
# diagnoses_long_df_refined['ICD10_code'] = diagnoses_long_df_refined['Diagnosis'].apply(lambda x: x.split(' ')[0])
diagnoses_long_df_refined['ICD10_code'] = diagnoses_long_df_refined['Diagnosis']


# Count occurrences and filter codes as before
code_counts_refined = diagnoses_long_df_refined['ICD10_code'].value_counts()
print(len(code_counts_refined),"# all code_counts")
codes_to_keep_refined = code_counts_refined[code_counts_refined >= K_value].index
print(len(codes_to_keep_refined),"# codes_to_keep_refined")
# Filter diagnoses to keep only frequently occurring codes
filtered_diagnoses_refined = diagnoses_long_df_refined[diagnoses_long_df_refined['ICD10_code'].isin(codes_to_keep_refined)]

# Apply one-hot encoding on the filtered ICD10 codes, grouping by the original index to align with the original dataframe
one_hot_encoding_refined = pd.get_dummies(filtered_diagnoses_refined['ICD10_code'],dummy_na=False).groupby(level=0).max()

# Merge with the original dataframe excluding the 'Diagnoses - ICD10' column
optimized_df_without_extra_index = df.drop(columns=['Diagnoses - ICD10']).join(one_hot_encoding_refined)

optimized_df_without_extra_index.drop(columns=[""],errors="ignore",inplace=True)
# Verify the transformation by displaying the first few rows
optimized_df_without_extra_index

## the empty "" col is cases with 0 diagnoses = all nans . it is ok ! 

* there's still rows with all nans? why that vs 0?

In [ ]:
# optimized_df_without_extra_index[optimized_df_without_extra_index[""]>0]
# df.loc[df["Participant ID"]==<REDACTED_PARTICIPANT_ID>]

In [ ]:
optimized_df_without_extra_index.columns

In [ ]:
optimized_df_without_extra_index.Sex = optimized_df_without_extra_index.Sex =="Male"

optimized_df_without_extra_index = optimized_df_without_extra_index.rename(columns={'Participant ID':"id"}).set_index("id")

In [ ]:
optimized_df_without_extra_index.to_parquet("df_diag_codes_flat2k.parquet")

In [ ]:
optimized_df_without_extra_index